
# Final Project 1
## End-to-End Unsupervised Learning Pipeline

### Professional Customer Segmentation Analysis

**Workflow**
- Business Understanding
- Data Audit
- Exploratory Data Analysis
- Feature Engineering
- Feature Scaling
- PCA
- KMeans Optimization
- Hierarchical Clustering
- DBSCAN
- Cluster Evaluation
- Cluster Profiling
- Business Insights



## Business Understanding

The objective is to identify meaningful customer segments using unsupervised learning techniques. These segments can support targeted marketing, customer retention, and personalized business strategies.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')


## Data Loading

In [ ]:
df = pd.read_csv('store_customers.csv')
df.head()

## Dataset Shape

In [ ]:
pd.DataFrame({'Rows':[df.shape[0]],'Columns':[df.shape[1]]})

## Dataset Information

In [ ]:
df.info()

## Missing Value Audit

In [ ]:
pd.DataFrame({'Missing':df.isnull().sum(),'Percentage':round(df.isnull().sum()/len(df)*100,2)})

## Statistical Summary

In [ ]:
df.describe(include='all')

## Duplicate Analysis

In [ ]:
pd.DataFrame({'Duplicate Records':[df.duplicated().sum()]})

## Gender Distribution

In [ ]:
plt.figure(figsize=(8,5)); sns.countplot(data=df,x='Gender'); plt.show()

## Age Distribution

In [ ]:
plt.figure(figsize=(8,5)); sns.histplot(df['Age'],kde=True,bins=25); plt.show()

## Income Distribution

In [ ]:
plt.figure(figsize=(8,5)); sns.histplot(df['Annual Income (k$)'],kde=True,bins=25); plt.show()

## Spending Score Distribution

In [ ]:
plt.figure(figsize=(8,5)); sns.histplot(df['Spending Score (1-100)'],kde=True,bins=25); plt.show()

## Pairplot

In [ ]:
sns.pairplot(df,hue='Gender'); plt.show()

## Correlation Matrix

In [ ]:
plt.figure(figsize=(8,6)); sns.heatmap(df.select_dtypes(include=np.number).corr(),annot=True,cmap='coolwarm'); plt.show()

## Outlier Assessment

In [ ]:
plt.figure(figsize=(10,5)); sns.boxplot(data=df[['Age','Annual Income (k$)','Spending Score (1-100)']]); plt.show()

## Encoding and Scaling

In [ ]:
encoder=LabelEncoder(); df['Gender']=encoder.fit_transform(df['Gender']); X=df.drop('CustomerID',axis=1); scaler=StandardScaler(); X_scaled=scaler.fit_transform(X)

## PCA Variance Analysis

In [ ]:
pca_full=PCA(); pca_full.fit(X_scaled); pd.DataFrame({'Component':range(1,len(pca_full.explained_variance_ratio_)+1),'Explained Variance':pca_full.explained_variance_ratio_,'Cumulative Variance':np.cumsum(pca_full.explained_variance_ratio_)})

## Cumulative Variance Plot

In [ ]:
plt.figure(figsize=(8,5)); plt.plot(np.cumsum(pca_full.explained_variance_ratio_),marker='o'); plt.show()

## PCA Projection

In [ ]:
pca=PCA(n_components=2); X_pca=pca.fit_transform(X_scaled); plt.figure(figsize=(8,6)); plt.scatter(X_pca[:,0],X_pca[:,1]); plt.show()

## KMeans Optimization

In [ ]:
inertia=[]; sil=[]
for k in range(2,11):
 m=KMeans(n_clusters=k,random_state=42,n_init=10)
 l=m.fit_predict(X_scaled)
 inertia.append(m.inertia_)
 sil.append(silhouette_score(X_scaled,l))

## Elbow Method

In [ ]:
plt.figure(figsize=(8,5)); plt.plot(range(2,11),inertia,marker='o'); plt.show()

## Silhouette Method

In [ ]:
plt.figure(figsize=(8,5)); plt.plot(range(2,11),sil,marker='o'); plt.show()

## KMeans Model

In [ ]:
kmeans=KMeans(n_clusters=5,random_state=42,n_init=10); kmeans_labels=kmeans.fit_predict(X_scaled)

## Hierarchical Clustering

In [ ]:
hier=AgglomerativeClustering(n_clusters=5); hier_labels=hier.fit_predict(X_scaled)

## Dendrogram

In [ ]:
linked=linkage(X_scaled,method='ward'); plt.figure(figsize=(12,5)); dendrogram(linked); plt.show()

## DBSCAN

In [ ]:
dbscan=DBSCAN(eps=0.6,min_samples=5); dbscan_labels=dbscan.fit_predict(X_scaled)

## Model Evaluation

In [ ]:
k_score=silhouette_score(X_scaled,kmeans_labels); h_score=silhouette_score(X_scaled,hier_labels); d_score=silhouette_score(X_scaled,dbscan_labels) if len(set(dbscan_labels))>1 else -1; pd.DataFrame({'Algorithm':['KMeans','Hierarchical','DBSCAN'],'Silhouette':[k_score,h_score,d_score]})

## KMeans Visualization

In [ ]:
plt.figure(figsize=(8,6)); plt.scatter(X_pca[:,0],X_pca[:,1],c=kmeans_labels); plt.show()

## Hierarchical Visualization

In [ ]:
plt.figure(figsize=(8,6)); plt.scatter(X_pca[:,0],X_pca[:,1],c=hier_labels); plt.show()

## DBSCAN Visualization

In [ ]:
plt.figure(figsize=(8,6)); plt.scatter(X_pca[:,0],X_pca[:,1],c=dbscan_labels); plt.show()

## Cluster Profiling

In [ ]:
profile=df.copy(); profile['Cluster']=kmeans_labels; profile.groupby('Cluster')[['Age','Annual Income (k$)','Spending Score (1-100)']].mean().round(2)

## Export Results

In [ ]:
profile.to_csv('cluster_labels.csv',index=False); profile.groupby('Cluster')[['Age','Annual Income (k$)','Spending Score (1-100)']].mean().round(2).to_csv('cluster_profiles.csv'); print('Files Saved')


# Business Recommendations

- High-income high-spending customers should receive premium offers.
- Medium-spending customers can be targeted through loyalty programs.
- Low-spending customers may respond to discounts and promotional campaigns.
- Segmentation enables efficient marketing resource allocation.



# Final Conclusion

A complete unsupervised learning pipeline was implemented. Multiple clustering algorithms were compared, customer groups were identified and evaluated, and actionable business insights were generated from the discovered segments.
